# 🦀 Day 6: Streams — Async Iterators

---

## What are Streams?

A **Stream** is the async version of an **Iterator**. Instead of `next()` returning `Option<T>` synchronously, it returns `Option<T>` asynchronously via `poll_next()`.

Use streams when you need to process items as they arrive over time.

In [ ]:
:dep tokio = { version = "1", features = ["full"] }
:dep tokio-stream = "0.1"
:dep futures = "0.3"

use tokio_stream::StreamExt;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // Create stream from iterator
    let mut stream = tokio_stream::iter(vec![1, 2, 3, 4, 5]);

    while let Some(n) = stream.next().await {
        println!("Got: {}", n);
    }
});

## StreamExt — map, filter, take, collect

Stream adapters work like iterator adapters, but are async.

In [ ]:
use tokio_stream::StreamExt;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let stream = tokio_stream::iter(1..=10)
        .map(|x| x * 2)
        .filter(|x| *x > 5)
        .take(5);

    let result: Vec<_> = stream.collect().await;
    println!("Collected: {:?}", result);
});

## ReceiverStream — Wrap mpsc Receiver as Stream

Turn an `mpsc::Receiver` into a stream for easy consumption.

In [ ]:
use tokio::sync::mpsc;
use tokio_stream::wrappers::ReceiverStream;
use tokio_stream::StreamExt;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let (tx, rx) = mpsc::channel(5);

    tokio::spawn(async move {
        tx.send(10).await.unwrap();
        tx.send(20).await.unwrap();
        tx.send(30).await.unwrap();
    });

    let mut stream = ReceiverStream::new(rx);
    let mut vec = Vec::new();
    while let Some(v) = stream.next().await {
        vec.push(v);
    }
    println!("From stream: {:?}", vec);
});

## IntervalStream — Time-Based Stream

Emit values at regular intervals.

In [ ]:
use tokio::time::{interval, Duration};
use tokio_stream::wrappers::IntervalStream;
use tokio_stream::StreamExt;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let interval = interval(Duration::from_millis(100));
    let mut stream = IntervalStream::new(interval);

    for _ in 0..3 {
        stream.next().await;
        println!("tick");
    }
});

## Merging Streams

Combine multiple streams into one with `futures::stream::select` or `merge`.

In [ ]:
use futures::stream::{self, StreamExt};

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let s1 = tokio_stream::iter(vec![1, 2, 3]);
    let s2 = tokio_stream::iter(vec![4, 5, 6]);

    let merged = stream::select(
        s1.map(|x| format!("A{}", x)),
        s2.map(|x| format!("B{}", x)),
    );

    let result: Vec<_> = merged.collect().await;
    println!("Merged (order may vary): {:?}", result);
});

## 🏋️ Exercise: Process a Stream of Sensor Readings

Create a stream of "sensor readings" (e.g., random or sequential numbers). Use `map` to convert to a struct or format, `filter` to keep only values above a threshold, and `take` to process the first 5. Collect and print the result.

In [ ]:
use tokio_stream::StreamExt;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // YOUR CODE HERE
    // 1. Create stream of sensor readings (e.g. tokio_stream::iter(1..=20))
    // 2. filter readings > 10
    // 3. take(5)
    // 4. collect and print
});

## 🎯 Key Takeaways

1. **Stream** = async Iterator; use `StreamExt::next()` to consume
2. `tokio_stream::iter()` — create stream from iterator
3. `ReceiverStream` — wrap mpsc receiver as stream
4. `IntervalStream` — time-based stream
5. `map`, `filter`, `take`, `collect` work on streams
6. Use `futures::stream::select` to merge streams

---

**Tomorrow:** Mini-project — Async chat server (TCP) 💬